In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pyomo.environ as pyo
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from pull_prices import merged_df_clean, merged_df_spike
from params import nodes, mcp, mdp, e, fee

# Create output folder
BASE_DIR = os.getcwd()
PLOT_DIR = os.path.join(BASE_DIR, "results", "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

print("Plots will be saved in:", PLOT_DIR)

In [ ]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 32)
        self.tr = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(32, 4, batch_first=True), 2
        )
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        return self.out(self.tr(self.fc(x)))


def train_transformer(df):

    if df is None:
        raise ValueError("train_transformer received None")

    df = df.copy()

    if "SP15" not in df.columns:
        raise KeyError("SP15 column missing")

    # EWMA-based expected behavior
    df["expected_price"] = df["SP15"].ewm(span=24, adjust=False).mean()

    return df

In [ ]:
def prepare_timeseries(df):

    # average price across nodes (simple fix)
    df_ts = df.groupby("datetime").agg({
        "SP15": "mean",
        "NonSpin": "mean",
        "RegDown": "mean",
        "RegUp": "mean",
        "Spin": "mean"
    }).reset_index()

    df_ts = df_ts.sort_values("datetime").reset_index(drop=True)

    return df_ts

In [ ]:
# =========================
# 🔹 GRAPH (LEARNED)
# =========================
def build_graph(df):

    pivot = df.pivot(index="datetime", columns="node", values="SP15")
    pivot = pivot.ffill()

    A = pivot.corr().values
    np.fill_diagonal(A, 0)

    D = np.diag(A.sum(axis=1))
    L = D - A

    return L

In [ ]:
def compute_anomaly(df):
    df = df.copy()

    if "expected_price" not in df.columns:
        raise KeyError("expected_price missing — run train_vae first")

    X = df[["SP15"]].values

    iso = IsolationForest(contamination=0.05)
    lof = LocalOutlierFactor()

    df["iso"] = (iso.fit_predict(X) == -1).astype(int)
    df["lof"] = (lof.fit_predict(X) == -1).astype(int)

    df["res"] = abs(df["SP15"] - df["expected_price"])

    scaler = MinMaxScaler()
    df[["res"]] = scaler.fit_transform(df[["res"]])

    df["anomaly"] = df["res"] + df["iso"] + df["lof"]
    df["anomaly"] = MinMaxScaler().fit_transform(df[["anomaly"]])

    return df

In [ ]:
def classify_anomaly(df, window=6, persist_threshold=2):
    df = df.copy()

    # rate of change
    df["roc"] = df["SP15"].diff().abs()
    df["roc"] = df["roc"] / (df["SP15"].shift(1).abs() + 1e-8)

    # persistence — how many surrounding hours are also anomalous
    df["anom_binary"]  = (df["anomaly"] > 0.5).astype(int)
    df["persistence"]  = (
        df["anom_binary"]
        .rolling(window, center=True, min_periods=1)
        .sum()
    )

    # local z-score
    df["local_mean"] = df["SP15"].rolling(window, center=True, min_periods=1).mean()
    df["local_std"]  = df["SP15"].rolling(window, center=True, min_periods=1).std().fillna(1)
    df["z_score"]    = (df["SP15"] - df["local_mean"]) / df["local_std"]

    # mean reversion — does price come back down after the spike?
    df["future_mean"] = df["SP15"].shift(-window).rolling(window, min_periods=1).mean()
    df["reverts"]     = (df["future_mean"] < df["SP15"]).astype(int)

    # genuine if: persists + reverts + not an instant jump
    df["is_genuine"] = (
        (df["persistence"] >= persist_threshold) &
        (df["reverts"] == 1) &
        (df["roc"] < 0.5)
    ).astype(int)

    # boost genuine spikes, suppress noise
    df["anomaly_adjusted"] = df["anomaly"] * np.where(df["is_genuine"], 1.5, 0.3)
    df["anomaly_adjusted"] = df["anomaly_adjusted"].clip(0, 1)

    return df

In [ ]:
def optimize(df, L, anomaly_on=True):
    df = df.copy()
    df["anomaly"] = df["anomaly"].fillna(0).clip(0, 1)

    anomaly_sensitivity = 0.2

    model = pyo.ConcreteModel()
    T = len(df)

    model.t = pyo.RangeSet(0, T - 1)
    model.n = pyo.Set(initialize=nodes)

    model.price = pyo.Param(model.t, initialize=lambda m, t: float(df["SP15"].iloc[t]))
    model.anom  = pyo.Param(model.t, initialize=lambda m, t: float(df["anomaly"].iloc[t]))

    model.buy  = pyo.Var(model.t, model.n, bounds=(0, mcp))
    model.sell = pyo.Var(model.t, model.n, bounds=(0, 1.2 * mdp))
    model.soc  = pyo.Var(model.t, bounds=(0, mcp))

    # --- SOC BALANCE ---
    def soc_rule(m, t):
        if t == 0:
            return m.soc[t] == mcp
        return m.soc[t] == m.soc[t-1] \
            + sum(m.buy[t, n] for n in m.n) * e \
            - sum(m.sell[t, n] for n in m.n) / e

    model.soc_c = pyo.Constraint(model.t, rule=soc_rule)

    # --- CAN'T SELL MORE THAN WHAT'S IN THE BATTERY ---
    def sell_energy_rule(m, t, n):
        if t == 0:
            return sum(m.sell[t, n2] for n2 in m.n) == 0
        return sum(m.sell[t, n2] for n2 in m.n) / e <= m.soc[t - 1]

    model.sell_energy_c = pyo.Constraint(model.t, model.n, rule=sell_energy_rule)

    # --- CAN'T CHARGE MORE THAN REMAINING HEADROOM ---
    def buy_headroom_rule(m, t, n):
        if t == 0:
            return pyo.Constraint.Skip
        return sum(m.buy[t, n2] for n2 in m.n) * e <= mcp - m.soc[t - 1]

    model.buy_headroom_c = pyo.Constraint(model.t, model.n, rule=buy_headroom_rule)

    # --- SELL LIMIT (anomaly-aware cap) ---
    def sell_limit_rule(m, t, n):
        if anomaly_on:
            return m.sell[t, n] <= mdp * (1 + anomaly_sensitivity * m.anom[t])
        return m.sell[t, n] <= mdp

    model.sell_limit = pyo.Constraint(model.t, model.n, rule=sell_limit_rule)

    # --- OBJECTIVE (anomaly scales attractiveness, fee applied per trade) ---
    def obj(m):
        return sum(
            m.sell[t, n] * m.price[t] * (1 + anomaly_sensitivity * m.anom[t]) - fee
            - m.buy[t, n] * (m.price[t] * (1 + anomaly_sensitivity * m.anom[t]) + fee)
            for t in m.t for n in m.n
        )

    model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

    # --- SOLVE WITH TERMINATION CHECK ---
    solver_result = pyo.SolverFactory("highs").solve(model)

    if solver_result.solver.termination_condition != pyo.TerminationCondition.optimal:
        print(f"[WARNING] Solver termination: {solver_result.solver.termination_condition}")

    soc    = [pyo.value(model.soc[t]) for t in model.t]
    profit = pyo.value(model.obj) or 0.0

    return profit, soc

In [ ]:
# =========================
# 🔹 PLOTTING (SAVE ONLY)
# =========================

def plot_price_comparison(df, name):

    plt.figure(figsize=(12,5))
    plt.plot(df["SP15"].values, label="Actual Price")
    plt.plot(df["expected_price"].values, label="Expected Price")

    plt.title("Price vs Expected Price")
    plt.legend()
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_price.png")
    plt.savefig(filepath)
    plt.close()


def plot_soc_comparison(base_soc, anom_soc, name):

    plt.figure(figsize=(12,5))
    plt.plot(base_soc, label="Baseline SOC")
    plt.plot(anom_soc, label="Anomaly SOC")

    plt.title("SOC Comparison")
    plt.legend()
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_soc.png")
    plt.savefig(filepath)
    plt.close()


def plot_volatility(df, name):

    df = df.copy()
    df["volatility"] = df["SP15"].rolling(24).std()

    plt.figure(figsize=(12,4))
    plt.plot(df["volatility"], color="purple")

    plt.title("Volatility")
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_volatility.png")
    plt.savefig(filepath)
    plt.close()


def plot_profit(c_base, c_anom, a_base, a_anom):

    labels = ["Clean Base", "Clean Anom", "Attack Base", "Attack Anom"]
    values = [c_base, c_anom, a_base, a_anom]

    plt.figure(figsize=(8,5))
    plt.bar(labels, values)

    plt.title("Profit Comparison")
    plt.grid(axis="y")

    filepath = os.path.join(PLOT_DIR, "profit_comparison.png")
    plt.savefig(filepath)
    plt.close()

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=1, latent_dim=8):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, 16)
        self.fc3 = nn.Linear(16, input_dim)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return self.fc3(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

In [ ]:
def train_vae(df):
    # NOTE: Transformer defined here but not used in pipeline.
    # train_vae (VAE) is used instead. Keep for future experimentation.

    df = df.copy()

    data = df["SP15"].values.reshape(-1, 1)
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)

    x = torch.tensor(data_scaled, dtype=torch.float32)

    model = VAE()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    def loss_fn(recon, x, mu, logvar):
        recon_loss = nn.MSELoss()(recon, x)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon_loss + kl_loss

    # Training
    for epoch in range(50):
        _ = epoch
        recon, mu, logvar = model(x)
        loss = loss_fn(recon, x, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Reconstruction
    recon, _, _ = model(x)
    recon_np = recon.detach().numpy()

    df["expected_price"] = scaler.inverse_transform(recon_np)

    return df

In [ ]:
def run(df):
    df_graph = df.copy()
    df_ts = prepare_timeseries(df)

    df_ts = train_vae(df_ts)
    df_ts = compute_anomaly(df_ts)
    df_ts = classify_anomaly(df_ts)          # ✅ NEW

    df_ts["anomaly"] = df_ts["anomaly_adjusted"]   # ✅ NEW — use refined score

    L = build_graph(df_graph)

    base_profit, base_soc = optimize(df_ts, L, False)
    anom_profit, anom_soc = optimize(df_ts, L, True)

    return df_ts, base_profit, anom_profit, base_soc, anom_soc

In [ ]:
print("Checking data...")

print("merged_df_clean type:", type(merged_df_clean))
print("merged_df_spike type:", type(merged_df_spike))

if merged_df_clean is None:
    raise ValueError("merged_df_clean is None")

if merged_df_spike is None:
    raise ValueError("merged_df_spike is None")

print("Clean shape:", merged_df_clean.shape)
print("Spike shape:", merged_df_spike.shape)

print("\nSample data:")
print(merged_df_clean.head())

In [ ]:
# =========================
# 🔹 EXECUTION
# =========================

print("Running CLEAN scenario...")

clean_df, c_base, c_anom, c_soc_base, c_soc_anom = run(merged_df_clean)

print("Running ATTACK scenario...")
attack_df, a_base, a_anom, a_soc_base, a_soc_anom = run(merged_df_spike)

print("\n===== RESULTS =====")
print("Clean:", c_base, c_anom)
print("Attack:", a_base, a_anom)

improvement_pct = ((a_anom - a_base) / abs(a_base)) * 100 if a_base else 0
print(f"Improvement: {improvement_pct:.2f}%")

# Save plots
plot_price_comparison(clean_df, "clean")
plot_volatility(clean_df, "clean")
plot_soc_comparison(c_soc_base, c_soc_anom, "clean")

plot_price_comparison(attack_df, "attack")
plot_volatility(attack_df, "attack")
plot_soc_comparison(a_soc_base, a_soc_anom, "attack")

plot_profit(c_base, c_anom, a_base, a_anom)

print(f"\nPlots saved in: {PLOT_DIR}")

In [ ]:
# =========================
# 🔹 OPTIONAL: SAVE RESULTS
# =========================

import os

RESULT_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)


# ---- 1. Save Profit Summary ----
results_df = pd.DataFrame({
    "scenario": ["clean_base", "clean_anom", "attack_base", "attack_anom"],
    "profit": [c_base, c_anom, a_base, a_anom]
})

results_path = os.path.join(RESULT_DIR, "results_summary.csv")
results_df.to_csv(results_path, index=False)


# ---- 2. Save SOC (Clean) ----
soc_clean_df = pd.DataFrame({
    "baseline_soc": c_soc_base,
    "anomaly_soc": c_soc_anom
})

soc_clean_path = os.path.join(RESULT_DIR, "soc_clean.csv")
soc_clean_df.to_csv(soc_clean_path, index=False)


# ---- 3. Save SOC (Attack) ----
soc_attack_df = pd.DataFrame({
    "baseline_soc": a_soc_base,
    "anomaly_soc": a_soc_anom
})

soc_attack_path = os.path.join(RESULT_DIR, "soc_attack.csv")
soc_attack_df.to_csv(soc_attack_path, index=False)


# ---- 4. Save Processed Data (with anomaly + expected) ----
clean_df.to_csv(os.path.join(RESULT_DIR, "clean_processed.csv"), index=False)
attack_df.to_csv(os.path.join(RESULT_DIR, "attack_processed.csv"), index=False)


# ---- 5. Save Volatility ----
clean_df["volatility"] = clean_df["SP15"].rolling(24).std()
attack_df["volatility"] = attack_df["SP15"].rolling(24).std()

clean_df[["datetime", "volatility"]].to_csv(
    os.path.join(RESULT_DIR, "clean_volatility.csv"), index=False
)
attack_df[["datetime", "volatility"]].to_csv(
    os.path.join(RESULT_DIR, "attack_volatility.csv"), index=False
)